# 01 — Première exploration des données DVF

Données : *Demandes de valeurs foncières géolocalisées* (data.gouv.fr), ventes immobilières
de la région PACA (départements 04, 05, 06, 13, 83, 84), années 2021 à 2025.

Les fichiers sont téléchargés avec `python src/download_data.py`.

## Chargement

In [ ]:
import pandas as pd
from pathlib import Path

# Remonter les dossiers jusqu'à trouver data/raw (fonctionne où que soit le notebook)
racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

fichiers = sorted((racine / "data" / "raw").glob("*.csv.gz"))
print(f"{len(fichiers)} fichiers trouvés dans {racine / 'data' / 'raw'}")

# Les codes (postal, commune, département, parcelle) sont des identifiants, pas des nombres :
# on les lit comme du texte pour garder les zéros de tête ("04" et non 4)
types_texte = {"code_postal": str, "code_commune": str, "code_departement": str, "id_parcelle": str}
df = pd.concat(
    (pd.read_csv(f, low_memory=False, dtype=types_texte) for f in fichiers),
    ignore_index=True,
)
print(f"Lignes brutes : {len(df):,}")
df.shape

## Aperçu et colonnes

In [ ]:
df.head()

In [ ]:
df.columns.tolist()

> ✍️ **À toi** — décris en quelques lignes les grandes familles de colonnes (la vente, l'adresse, le bien, le terrain, les lots, le GPS) et indique quelle colonne sera la variable à prédire.
>
> *(remplace ce bloc par ton commentaire)*

## Types de ventes et de biens

In [ ]:
df["nature_mutation"].value_counts()

In [ ]:
df["type_local"].value_counts(dropna=False)

## Une vente = plusieurs lignes ?

In [ ]:
lignes_par_vente = df.groupby("id_mutation").size()
print(f"Nombre de ventes uniques : {lignes_par_vente.shape[0]:,}")
print(f"Nombre de lignes         : {len(df):,}")
lignes_par_vente.value_counts().head(10)

In [ ]:
# Exemple concret : une vente qui occupe plusieurs lignes, avec le même prix répété
exemple = lignes_par_vente[lignes_par_vente.between(2, 4)].index[0]
df.loc[df["id_mutation"] == exemple,
       ["id_mutation", "date_mutation", "valeur_fonciere", "type_local", "surface_reelle_bati", "nature_culture"]]

> ✍️ **À toi** — explique avec tes mots pourquoi une même vente apparaît sur plusieurs lignes, et pourquoi c'est un piège si on calcule des prix moyens ou si on entraîne un modèle sur ces lignes telles quelles.
>
> *(remplace ce bloc par ton commentaire)*